### CatBoost

- CatBoost processes raw categorical text directly, completely eliminating the need for OneHotEncoder or TargetEncoder.

- No ColumnTransformer or preprocessing required, just pass the raw data and define text columns using cat_features.

- It uses Ordered Target Statistics (OTS), dynamically calculating category target averages sequentially to mathematically prevent data leakage.

- CatBoost natively routes numeric missing values (NaN) through its decision trees without crashing, so we skip imputation too.

- It automatically detects and merges related categorical features (day="Sunday" + color="Yellow") during tree splits to find complex patterns.

- It builds symmetric decision trees, which restricts structural complexity and makes the model incredibly stable on test data.

- Bundling features, labels and categories into a Pool object compiles the dataset into a highly optimized C++ memory block for fast training speeds.

---
## <u>Import functions and load dataset</u>

In [21]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor, Pool

data = sns.load_dataset("diamonds")

X = data.drop(columns=["price"])
y = data["price"]

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   x        53940 non-null  float64 
 7   y        53940 non-null  float64 
 8   z        53940 non-null  float64 
dtypes: category(3), float64(6)
memory usage: 2.6 MB


---
## <u>Handle categorical data</u>

In [17]:
# No need to use any encoders, just define the categorical features and clean missing values (nan) from data

cat_features = ["cut", "color", "clarity"]

# First we convert the cat_features into string so that we can replace "nan" with "missing"
for col in cat_features:
    X[col] = X[col].astype(str).replace('nan', 'Missing')

---
## <u>Train Test Split</u>

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## <u>Pooling</u>

In [23]:
# Pools Bundle the data, labels, and the list of categorical features into  single, highly optimized C++ memory block.
# This makes training exponentially faster.
train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

---
## <u>Create model</u>

In [25]:
cbr_model = CatBoostRegressor(
    iterations=200,      # 200 trees
    learning_rate=0.1,   
    max_depth=6,             
    random_state=42, 
    verbose=0
)

---
## <u>Train and Predict</u>

In [26]:
# Because we used Pools, we just pass the Pool object directly, No need to pass X and y separately
cbr_model.fit(train_pool)
y_test_pred = cbr_model.predict(test_pool)
y_train_pred = cbr_model.predict(train_pool)

---
## <u>Evaluate</u>

In [28]:
# check both Train and Test to monitor the Bias-Variance tradeoff (checking for overfitting)

print("for Catboost Regressor (baseline) :-")
print("\nTrain r2_score : ", r2_score(y_train, y_train_pred))
print("\nTest r2_score : ", r2_score(y_test, y_test_pred))

for Catboost Regressor (baseline) :-

Train r2_score :  0.9830570711997552

Test r2_score :  0.9815326986894902
